In [ ]:
import copy
import csv
import glob
import json
import librosa
import librosa.display
import random
import os
import sys
import torch
import ast  # for parsing stringified tuples in CSV


import IPython.display as ipd
from IPython.display import clear_output
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from collections import defaultdict
from scipy.io import wavfile
from scipy.spatial.distance import euclidean
from scipy import stats

sys.path.append("/om2/user/gelbanna/repos/projects/continuous-speech-recognition")

from front_end.cochlear_model import CochlearModel
from utils import load_yaml_config

#testing out cochlear model

In [ ]:
# Example instantiation
config = load_yaml_config("/om2/user/bjmedina/memory/utils/cochleagram_params.yaml")
config['frontend']['cochlear']['filter_params']['sr'] = 44100
model = CochlearModel(config.frontend.cochlear, device="cuda")


# Example forward pass
# x = [torch.randn(1, 1, 16000), torch.randn(1, 1, 8000), torch.randn(1, 1, 32000)]  # Example input tensor
# output = model(x[0].to("cuda"))
# print(output.shape)  # Example output shape 

In [ ]:
def plot_torch_cochleagram(output):
    cochleagram_dB = output.squeeze(0).squeeze(0).detach().cpu().numpy()
    
    plt.figure(figsize=(10, 6))
    plt.imshow(
        cochleagram_dB,
        origin='lower',
        aspect='auto',
        # extent=[time_bins[0].item(), time_bins[-1].item(),
        #         freq_bins[0].item(), freq_bins[-1].item()],
        cmap='viridis'
    )
    plt.colorbar(label='Amplitude [dB]')
    plt.title("Cochleagram")
    plt.xlabel("Time [s]")
    plt.ylabel("Frequency [Hz]")
    plt.tight_layout()
    plt.show()
    
def plot_torch_corrs(output):
    corrs = output.squeeze(0).squeeze(0).detach().cpu().numpy()
    
    plt.figure(figsize=(10, 6))
    plt.matshow(
        corrs, vmin=0, vmax=1
    )
    plt.colorbar(label='Pearson Correlation')
    plt.title("Segments")
    plt.xlabel("Segments")
    plt.ylabel("Segments")
    plt.tight_layout()
    plt.show()
# plot_torch_cochleagram(output)

In [ ]:
def rms_normalize(signal, target_rms=0.1):
    """
    Normalize a signal to a target RMS level.
    
    Parameters:
        signal (numpy array): The input signal.
        target_rms (float): The desired RMS level (default is 0.1).
    
    Returns:
        numpy array: The RMS-normalized signal.
    """
    rms = np.sqrt(np.mean(signal**2))
    if rms == 0:
        return signal  # Avoid division by zero
    return signal * (target_rms / rms)

In [ ]:
# grabbing stationarity scores for sounds that are in the in the eval set of AudioSet (which is what we typically use for memory experiment)
stationarity_scores_path = "/om2/user/bjmedina/chexture_choolbox/assets/OVERLAPPED_0.5s_eval_sound_list_with_stationarity_score_no_speech_no_music_audioset_matlab_coch_rms0p02.csv"
stationarity_scores_ = pd.read_csv(stationarity_scores_path)

In [ ]:
# grabbing the textures that were selected to be in the memory 
texture_pairs_paths = "/mindhive/mcdermott/www/mturk_stimuli/bjmedina/mem_exemplar_0.5s_adjacent/texture_filenames.json"

with open(texture_pairs_paths) as f:
    texture_pairs = json.load(f)

In [ ]:
ss_scores_to_texture = defaultdict(list)
times_to_texture     = defaultdict(list)

for texture_pair in texture_pairs:
    texture_id = "/".join(texture_pair['full_path'].split("/")[-2:])

    avg_ss_scores = stationarity_scores_[stationarity_scores_.filepath==texture_id].stationarity_score.tolist()
    times = stationarity_scores_[stationarity_scores_.filepath==texture_id].onset_time.tolist()
    
    ss_scores_to_texture[texture_id] = avg_ss_scores
    times_to_texture[texture_id]     = times

In [ ]:
# plot distribution of stationarity scores
avg_scores = []
for id in ss_scores_to_texture:
    avg_scores.extend(ss_scores_to_texture[id])

In [ ]:
plt.title("Stationarity Scores")
plt.axvline(x=np.mean(avg_scores), label=f"$\mu={np.mean(avg_scores)}$\n$\sigma={np.std(avg_scores)}$")
plt.hist(avg_scores, bins=100, alpha=0.5)
plt.legend()

avg_ss_score = np.mean(avg_scores)
std_ss_score = np.std(avg_scores)

In [ ]:
# consider how you screen for different sound
fps = stationarity_scores_[stationarity_scores_.stationarity_score < avg_ss_score].filepath.tolist(); fps = list(set(fps))

In [ ]:
base_dir = "/om2/data/public/audioset/wavs/eval_segments_downloads/"
offset = 0.1

#save_dir = f"/om2/user/bjmedina/data/texture_pairs/cochleagram_norms_ratio-max_stationarity-lessthan{avg_ss_score:.2f}-offset{offset}/"


In [ ]:
import itertools

def apply_linear_ramp(signal, sample_rate, ramp_duration_ms=5):
    """
    Applies a linear ramp (fade-in and fade-out) to a signal over a given duration.

    Args:
        signal (numpy.ndarray): Input audio signal (1D NumPy array).
        sample_rate (int): Sampling rate of the audio signal.
        ramp_duration_ms (float): Duration of the ramp in milliseconds (default: 5ms).

    Returns:
        numpy.ndarray: Signal with applied linear ramp.
    """
    num_samples = len(signal)
    ramp_samples = int((ramp_duration_ms / 1000) * sample_rate)  # Convert ms to samples

    if ramp_samples * 2 >= num_samples:
        raise ValueError("Ramp duration too long for the signal length!")

    # Create fade-in and fade-out ramps
    fade_in = np.linspace(0, 1, ramp_samples)
    fade_out = np.linspace(1, 0, ramp_samples)

    # Apply ramps to the signal
    signal[:ramp_samples] *= fade_in  # Apply fade-in
    signal[-ramp_samples:] *= fade_out  # Apply fade-out

    return signal

import scipy.signal

def compute_autocorrelation(signal):
    signal = signal - signal.mean()  # remove DC offset
    corr = torch.nn.functional.conv1d(
        signal[None, None, :],
        signal[None, None, :].flip(-1),  # cross-correlation
        padding=signal.numel() - 1
    )[0, 0]
    #corr = corr / corr.abs().max()  # normalize
    mid = corr.numel() // 2
    return corr[mid:]  # keep only non-negative lags

from scipy.signal import find_peaks

def is_pure_tone(corr, sr, min_freq=100, max_freq=3000):
    min_lag = int(sr / max_freq)
    max_lag = int(sr / min_freq)

    search_region = corr[min_lag:max_lag].cpu().numpy()
    peaks, props = find_peaks(search_region, height=0.3)

    if len(peaks) == 0:
        return False, 0.0

    first_peak = props["peak_heights"][0]
    peak_std = np.std(props["peak_heights"])

    # A pure tone has a strong first peak and little variation
    is_tone = (first_peak > 0.5) and (peak_std < 0.1)
    return is_tone, first_peak


import torch

def compute_autocorrelation_torch(signal):
    """Compute normalized autocorrelation using PyTorch."""
    signal = signal - torch.mean(signal)  # remove DC
    corr = torch.nn.functional.conv1d(
        signal[None, None, :],
        signal.flip(0)[None, None, :],
        padding=signal.shape[0]-1
    )[0, 0]
    corr = corr / torch.max(torch.abs(corr))  # normalize
    mid = len(corr) // 2
    return corr[mid:]  # keep only non-negative lags

def should_filter_out(data_np, sr, min_freq=100, max_freq=3000, threshold=0.5, std_thresh=0.1, device="cuda"):
    """
    Returns True if a signal is likely a pure tone.
    """
    # Convert to torch and move to GPU
    data = torch.tensor(data_np, dtype=torch.float32, device=device)

    # Compute autocorrelation
    corr = compute_autocorrelation_torch(data)

    # Convert frequency to lag range
    min_lag = int(sr / max_freq)
    max_lag = int(sr / min_freq)

    if max_lag >= len(corr):
        return False  # signal too short

    segment = corr[min_lag:max_lag]

    # Find peaks (using simple argmax as proxy; optional: write peak finder in torch)
    peak_vals, peak_idxs = torch.topk(segment, k=min(5, len(segment)))

    if len(peak_vals) == 0:
        return False

    first_peak = peak_vals[0].item()
    peak_std = torch.std(peak_vals).item()

    return (first_peak > threshold) and (peak_std < std_thresh), first_peak, peak_std

In [ ]:
from tqdm.notebook import tqdm

# target_sr = 22050*2

# k = 0

# pure_tone_dict = defaultdict(list)
# with tqdm(total=len(fps), file=sys.stdout) as pbar:
#     for curr_sound_idx, curr_sound in enumerate(fps):
    
#         # Compute cochleagram for each segment
#         cochleagrams = []
#         time_averaged_specs = []
    
#         #stationarity, times = ss_scores_to_texture[curr_sound], times_to_texture[curr_sound]
        
#         audio_path = base_dir + curr_sound
        
#         data, samplerate = librosa.load(audio_path)
                    
#         get_out = should_filter_out(data, samplerate)
        
#         pure_tone_dict[curr_sound] = [get_out]
#         pbar.update(1)
#         #sleep(1)

device = "cuda" if torch.cuda.is_available() else "cpu"

pure_tone_dict = defaultdict(list)

with tqdm(total=len(fps), file=sys.stdout) as pbar:
    for curr_sound in fps:
        audio_path = base_dir + curr_sound
        data, samplerate = librosa.load(audio_path)

        should_filter, first_peak, peak_std  = should_filter_out(data, samplerate, device=device)
        pure_tone_dict[curr_sound] = [should_filter, first_peak, peak_std]

        pbar.update(1)

In [ ]:
def plot_autocorrelation_peaks(data_np, sr, min_freq=100, max_freq=3000, device="cuda"):
    data = torch.tensor(data_np, dtype=torch.float32, device=device)
    corr = compute_autocorrelation_torch(data).cpu().numpy()

    # Convert frequency range to lags
    min_lag = int(sr / max_freq)
    max_lag = int(sr / min_freq)

    if max_lag >= len(corr):
        print("Signal too short for autocorrelation range.")
        return

    segment = corr[min_lag:max_lag]
    lags = torch.arange(min_lag, max_lag).cpu().numpy()

    # Top 5 peaks
    segment_torch = torch.tensor(segment)
    peak_vals, peak_idxs = torch.topk(segment_torch, k=min(5, len(segment_torch)))

    # Plot full autocorr segment
    plt.figure(figsize=(10, 4))
    plt.plot(lags, segment, label='Autocorrelation')
    plt.scatter(lags[peak_idxs], peak_vals.numpy(), color='red', label='Top Peaks')

    # Annotate std and first peak
    peak_std = torch.std(peak_vals).item()
    first_peak = peak_vals[0].item()
    plt.title(f"First Peak: {first_peak:.3f} | Std of Peaks: {peak_std:.3f}")
    plt.xlabel("Lag (samples)")
    plt.ylim([-1,1])
    plt.ylabel("Normalized Autocorr")
    plt.legend()
    plt.grid(True)
    plt.show()

In [ ]:
audio_path = base_dir + "eval_segments_downloads_0/YT_dt8eRF7lCt4.wav"
data, samplerate = librosa.load(audio_path)

plot_autocorrelation_peaks(data, samplerate, min_freq=100, max_freq=3000, device="cuda")

In [ ]:
audio_paths = glob.glob("/om2/user/bjmedina/data/texture_pairs/max_cochleagram_mse-time_avg_mse-25.0db-lowpassfiltered-10hz-lessthan-0.62-offset0.1-segment_duration0.5/all_pairs/*dt8eRF7lCt4*")
data, samplerate = librosa.load(audio_paths[0])
print(audio_paths[0])
plot_autocorrelation_peaks(data[len(data)//2:], samplerate, min_freq=100, max_freq=3000, device="cuda")
plt.plot(data[len(data)//2:], alpha=0.5)

In [ ]:
data, samplerate = librosa.load(audio_paths[1])
print(audio_paths[1])
plot_autocorrelation_peaks(data[:len(data)//2], samplerate, min_freq=100, max_freq=3000, device="cuda")
plt.plot(data[:len(data)//2], alpha=0.5)

In [ ]:
audio_paths = glob.glob("/om2/user/bjmedina/data/texture_pairs/max_cochleagram_mse-time_avg_mse-25.0db-lowpassfiltered-10hz-lessthan-0.62-offset0.1-segment_duration0.5/all_pairs/*dt8eRF7lCt4*")
data, samplerate = librosa.load(audio_paths[0])
plot_autocorrelation_peaks(data, samplerate, min_freq=100, max_freq=3000, device="cuda")

In [ ]:
plt.plot(data)

In [ ]:
pure_tone_dict_sorted = dict(sorted(pure_tone_dict.items(), key=lambda item: item[1][1]))

In [ ]:
def equal_width_bar(to_plot, data=None, num_bins=5, label=""):
    if data == None:
        bin_edges = np.linspace(min(to_plot), max(to_plot), num_bins + 1)  # Evenly spaced bins
    else:
        bin_edges = np.linspace(min(data), max(data), num_bins + 1)  # Evenly spaced bins
    
    # Compute histogram using numpy
    counts, edges = np.histogram(to_plot, bins=bin_edges, density=True)

    # Plot histogram manually with specified bin widths
    plt.bar(edges[:-1], counts, width=np.diff(edges), edgecolor='black', alpha=0.7, label=label)

In [ ]:
from IPython.display import Audio, display
num_clips = len(pure_tone_dict_sorted)
# Top 10 most voice-like
print("🔊 Top 10 clips with most pure-tone-like clips:")
for path, prob in list(pure_tone_dict_sorted.items())[-10:][::-1]:  # reverse so highest first
    print(f"{path} | ratio: {prob}")
    audio_path = base_dir + path
    data, samplerate = librosa.load(audio_path)
    print(f"Total length of clip: {len(data)/samplerate}(s)")
    display(Audio(audio_path))


# Middle 10 least voice-like
print("🔊 Middle 10 clips with most pure-tone-like clips:")
for path, prob in list(pure_tone_dict_sorted.items())[num_clips//2 - 5:num_clips//2 + 5]:
    print(f"{path} | ratio: {prob}")
    audio_path = base_dir + path
    data, samplerate = librosa.load(audio_path)
    print(f"Total length of clip: {len(data)/samplerate}(s)")
    display(Audio(audio_path))

# Bottom 10 least voice-like
print("🔊 Bottom 10 clips with most pure-tone-like clips:")
for path, prob in list(pure_tone_dict_sorted.items())[:10]:
    print(f"{path} | ratio: {prob}")
    audio_path = base_dir + path
    data, samplerate = librosa.load(audio_path)
    print(f"Total length of clip: {len(data)/samplerate}(s)")
    display(Audio(audio_path))

In [ ]:
save_dir = f"/om2/user/bjmedina/data/filters/pure_tone_like"
import os
import soundfile as sf

base_save_dir = "/om2/user/bjmedina/data/filters/pure_tone_like/saved_clips"
categories = ["top", "middle", "bottom"]

# Make the folders if they don't exist
for category in categories:
    os.makedirs(os.path.join(base_save_dir, category), exist_ok=True)

num_clips = len(pure_tone_dict_sorted)

def save_and_display_clip(category, index, path, prob):
    print(f"{path} | ratio: {prob}")
    audio_path = base_dir + path
    data, samplerate = librosa.load(audio_path)
    duration = len(data) / samplerate
    print(f"Total length of clip: {duration:.2f}(s)")

    # Save to category-specific subfolder
    filename = f"{index:02d}_{os.path.basename(path)}"
    save_path = os.path.join(base_save_dir, category, filename)
    sf.write(save_path, data, samplerate)

    # Display in notebook
    display(Audio(audio_path))


# Top 10 most pure-tone-like
print("🔊 Top 10 clips with most pure-tone-like content:")
for i, (path, prob) in enumerate(list(pure_tone_dict_sorted.items())[-10:][::-1]):
    save_and_display_clip("top", i, path, prob)

# Middle 10
print("\n🔊 Middle 10 clips:")
mid_start = num_clips // 2 - 5
mid_end = num_clips // 2 + 5
for i, (path, prob) in enumerate(list(pure_tone_dict_sorted.items())[mid_start:mid_end]):
    save_and_display_clip("middle", i, path, prob)

# Bottom 10 least pure-tone-like
print("\n🔊 Bottom 10 clips (least pure-tone-like):")
for i, (path, prob) in enumerate(list(pure_tone_dict_sorted.items())[:10]):
    save_and_display_clip("bottom", i, path, prob)